In [ ]:
import torch
import torch.nn as nn
from utils import intersection_over_union

### 2. Model Architecture

In [6]:
# Darknet Architectures Configuration
architectures_config = [
    # Tuple: (kernel_size, num_filters, stride, padding)
    (7, 64, 2, 3),
    "M", # MaxPool 2x2
    (3, 192, 1, 1),
    "M",
    (1, 128, 1, 0),
    (3, 256, 1, 1),
    (1, 256, 1, 0),
    (3, 512, 1, 1),
    "M",
    [(1, 256, 1, 0), (3, 512, 1, 1), 4],
    (1, 512, 1, 0),
    (3, 1024, 1, 1),
    "M",
    # List: Tuple 1, Tuple 2, Number repeat
    [(1, 512, 1, 0), (3, 1024, 1, 1), 2],
    (3, 1024, 1, 1),
    (3, 1024, 2, 1),
    (3, 1024, 1, 1),
    (3, 1024, 1, 1),
]

In [7]:
### CNNBLOCK ###
# X --> Z = W * X + b --> Z_batch = BatchNorm(Z) --> A = LeakyReLU(Z_batch)
def CNNBlock(in_channels, out_channels, kernel_size, stride, padding):
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, stride=stride, padding=padding),
        nn.BatchNorm2d(out_channels),
        nn.LeakyReLU(0.1)
    )

### FULLY CONNECTED LAYER ###
def create_fcs(split_size=7, num_boxes=2, num_classes=20):
    S, B, C = split_size, num_boxes, num_classes
    return nn.Sequential(
        nn.Flatten(),
        nn.Linear(1024 * S * S, 496),
        nn.LeakyReLU(0.1),
        nn.Dropout(0.5),
        nn.Linear(496, S * S * (B * 5 + C))
    )

In [8]:
### CREATE CONV LAYER ###
def create_conv_layer(in_channels, architectures):
    layers = []
    
    for x in architectures:
        if type(x) == tuple:
            layers += [CNNBlock(in_channels, x[1], kernel_size=x[0], stride=x[2], padding=x[3])]
            in_channels = x[1]

        elif type(x) == str:
            layers += [nn.MaxPool2d(kernel_size=2, stride=2)]

        elif type(x) == list:
            conv1 = x[0]
            conv2 = x[1]
            num_repeats = x[2]

            for _ in range(num_repeats):
                layers += [CNNBlock(in_channels, conv1[1], kernel_size=conv1[0], stride=conv1[2], padding=conv1[3])]
                in_channels = conv1[1]
                
                layers += [CNNBlock(conv1[1], conv2[1], kernel_size=conv2[0], stride=conv2[2], padding=conv2[3])]
                in_channels = conv2[1]

    return nn.Sequential(*layers)

In [ ]:
### YOLOv1 MODEL ###
def YOLOv1(in_channels=3, split_size=7, num_boxes=2, num_classes=80):
    # Backbone
    darknet_backbone = create_conv_layer(in_channels, architectures_config)

    # Head
    fcs_head = create_fcs(split_size, num_boxes, num_classes)
    
    # Model
    model = nn.Sequential(
        darknet_backbone,
        fcs_head
    )

    return model

### 3. Loss Function

In [ ]:
# config
lambda_noobj = 0.5
lambda_coord = 5

In [ ]:
# loss function
mse = nn.MSELoss(reduction="sum")

def yolo_loss(predictions, target, split_size=7, num_boxes=2, num_classes=80, lambda_noobj=0.5, lambda_coord=5):
    S, B, C = split_size, num_boxes, num_classes
    predictions = predictions.reshape(-1, S, S, C + B * 5)

    # Layout: [classes..., conf1, x1, y1, w1, h1, conf2, x2, y2, w2, h2]
    box1_start = C + 1
    box2_conf = C + 5
    box2_start = C + 6

    iou_b1 = intersection_over_union(
        predictions[..., box1_start:box1_start + 4],
        target[..., box1_start:box1_start + 4],
    )
    iou_b2 = intersection_over_union(
        predictions[..., box2_start:box2_start + 4],
        target[..., box1_start:box1_start + 4],
    )

    ious = torch.cat([iou_b1.unsqueeze(0), iou_b2.unsqueeze(0)], dim=0)
    _, bestbox = torch.max(ious, dim=0)

    exists_box = target[..., C].unsqueeze(3)

    # BOX LOSS
    bestbox = bestbox.unsqueeze(-1)

    box_predictions = exists_box * (
        bestbox * predictions[..., box2_start:box2_start + 4]
        + (1 - bestbox) * predictions[..., box1_start:box1_start + 4]
    )

    box_targets = exists_box * target[..., box1_start:box1_start + 4]

    box_predictions[..., 2:4] = (
        torch.sign(box_predictions[..., 2:4])
        * torch.sqrt(torch.abs(box_predictions[..., 2:4]) + 1e-6)
    )
    box_targets[..., 2:4] = torch.sqrt(box_targets[..., 2:4] + 1e-6)

    box_loss = mse(
        torch.flatten(box_predictions, end_dim=-2),
        torch.flatten(box_targets, end_dim=-2)
    )

    # OBJECT LOSS
    pred_box = (
        bestbox * predictions[..., box2_conf:box2_conf + 1]
        + (1 - bestbox) * predictions[..., C:C + 1]
    )

    object_loss = mse(
        torch.flatten(exists_box * pred_box),
        torch.flatten(exists_box * target[..., C:C + 1]),
    )

    # NO OBJECT LOSS
    no_object_loss = mse(
        torch.flatten((1 - exists_box) * predictions[..., C:C + 1], start_dim=1),
        torch.flatten((1 - exists_box) * target[..., C:C + 1], start_dim=1),
    )

    no_object_loss += mse(
        torch.flatten((1 - exists_box) * predictions[..., box2_conf:box2_conf + 1], start_dim=1),
        torch.flatten((1 - exists_box) * target[..., C:C + 1], start_dim=1),
    )

    # CLASS LOSS
    class_loss = mse(
        torch.flatten(exists_box * predictions[..., :C], end_dim=-2),
        torch.flatten(exists_box * target[..., :C], end_dim=-2),
    )

    # TOTAL LOSS
    loss = (
        lambda_coord * box_loss
        + object_loss
        + lambda_noobj * no_object_loss
        + class_loss
    )

    return loss